# 10 — Documentar tabelas Gold

Executar após 09. Atualiza apenas descrições das oito tabelas, sem reprocessar dados. Exige permissão para alterar metadados. Confira o catálogo antes de prosseguir.


In [0]:
# CMD 1 — catalogo=workspace: catálogo das tabelas Gold já homologadas.
dbutils.widgets.text('catalogo','workspace')
print('Preencha catalogo e depois execute os demais CMDs.')


In [0]:
# CMD 2 — Ler o catálogo de negócio versionado. Nenhuma escrita nesta célula.
from pathlib import Path
import json,re
catalogo=dbutils.widgets.get('catalogo').strip()
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',catalogo): raise ValueError('Catálogo inválido')
p=Path.cwd().resolve()
raiz=next(p for p in [p,*p.parents] if (p/'docs/gold_catalogo.json').is_file())
metadados=json.loads((raiz/'docs/gold_catalogo.json').read_text())
for nome,meta in metadados.items():
    print(nome,meta['grao'])
    spark.sql(f'DESCRIBE TABLE `{catalogo}`.`medallion_dev_ouro`.`{nome}`').collect()


In [0]:
# CMD 3 — Aplicar descrições de negócio e grão e verificar a releitura.
for nome,meta in metadados.items():
    tabela=f'`{catalogo}`.`medallion_dev_ouro`.`{nome}`'
    comentario=meta['comentario'].replace("'","''")
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    rows=spark.sql(f'DESCRIBE TABLE EXTENDED {tabela}').collect()
    observado=next((r['data_type'] for r in rows if r['col_name'].strip()=='Comment'),None)
    if observado!=meta['comentario']: raise ValueError('Comentário divergente: '+nome)
    print(nome,'DESCRIÇÃO E GRÃO: OK')
print('CATÁLOGO GOLD: OK')


Exporte com outputs e envie para revisão. Não é necessário reexecutar 06/07. Esta alteração é de metadados e pode ser repetida; não altera hashes dos registros dos snapshots.


Falta de uma descricao das colunas para dar clareza em nivel de negocio e tambem identificacao de colunas PK con visibilidade no unity catalog e tambem Tags